# grid_100x100 セマンティックブロック層デモ

**前提**: UE Editor で `grid_100x100` を開き **PIE 実行中** にこのノートを実行してください。

## シナリオ

1. 既存床・既存 `block_*` より十分高い位置に **仮床**（`BP_Floor_30x30` を 90×90 cm にスケール・3×3 マス）をスポーン
2. 隅 **5×5 = 25** マス（gx,gy = 1..5）を走査しラベル付け
3. ブロック下面を仮床上面 + **0.15 m** に敷き詰め（`sem_block_*`、着色なし）
   - **floor** → 透過 (F)、**air** → 実体 (T)
4. `.semantic_layer_registry.json` に保存

### ラベル規則（配置前）

- `z_initial` = 仮床上面 + 0.15 m で probe → 干渉ありなら **wall**
- なければ `z_initial − 0.30 m` で probe → 干渉ありなら **floor**、なければ **air**

### テスト期待

- 仮床内（1..3）→ **floor** ×9
- 仮床外（4..5）→ **air** ×16
- **wall** ×0（生成高度が正しければ仮床は wall にならない）

ロジック: `grid_env_10k_semantic.py` / `block_semantic_scan.py`

カーネル: `conda activate simworld`

In [ ]:
import importlib
import sys
from pathlib import Path
from typing import Optional

from simworld.communicator.unrealcv import UnrealCV


def _find_project_root() -> Path:
    for start in (Path.cwd().resolve(), Path(".").resolve()):
        for candidate in (start, *start.parents):
            if (candidate / "setup.py").exists() and (candidate / "simworld").is_dir():
                return candidate
    return Path.cwd().resolve().parent.parent


_root = _find_project_root()
_sem_dir = _root / "dev" / "grid_env_10k_semantic"
_g10k_dir = _root / "dev" / "grid_env_10k"
_geh_dir = _root / "dev" / "grid_env_hri"
for p in (_root, _sem_dir, _g10k_dir, _geh_dir):
    if str(p) not in sys.path:
        sys.path.insert(0, str(p))

ucv: Optional[UnrealCV] = None
print(f"[Paths] root={_root}")
print(f"[Paths] semantic={_sem_dir}")

In [ ]:
import grid_env_10k_semantic as sem

importlib.reload(sem)

ucv, _ = sem.ensure_connection()
if not ucv.client.isconnected():
    raise RuntimeError("UnrealCV not connected — start grid_100x100 PIE first.")
print("OK: UnrealCV connected")

In [ ]:
result = sem.run_semantic_layer_demo(ucv, cleanup_before=True)
counts = {"wall": 0, "floor": 0, "air": 0}
for s in result.semantics.values():
    counts[s] += 1
print(f"placed={len(result.blocks)} wall/floor/air={counts}")
print(f"registry={result.registry_path}")

In [ ]:
# 再実行前に sem_* Actor を削除する場合
# sem.cleanup_semantic_layer(ucv, result.blocks)